In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/ML/final/data/train.csv', low_memory = False)
print(df.shape)
df.head(5)

(100000, 28)


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


XỬ LÝ DỮ LIỆU RÁC VÀ FILL

In [ ]:
df.drop(columns = ['ID', 'Name', "SSN"], inplace = True, errors = 'ignore')


In [ ]:
def clean_numeric(series):
    """Xóa ký tự lỗi, chuyển về float, đặt âm/vô lý thành NaN."""
    s = series.astype(str).str.replace(r"[^0-9.\-]", "", regex=True)
    s = pd.to_numeric(s, errors="coerce")
    return s

NUMERIC_COLS = [
    "Age", "Annual_Income", "Monthly_Inhand_Salary", "Num_Bank_Accounts",
    "Num_Credit_Card", "Interest_Rate", "Num_of_Loan",
    "Delay_from_due_date", "Num_of_Delayed_Payment", "Changed_Credit_Limit",
    "Num_Credit_Inquiries", "Outstanding_Debt", "Credit_Utilization_Ratio",
    "Total_EMI_per_month", "Amount_invested_monthly", "Monthly_Balance",
]

for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = clean_numeric(df[col])

# Các giá trị âm vô lý → NaN
NON_NEGATIVE = [
    "Age", "Annual_Income", "Monthly_Inhand_Salary", "Num_Bank_Accounts",
    "Num_Credit_Card", "Num_of_Loan", "Num_of_Delayed_Payment",
    "Num_Credit_Inquiries", "Outstanding_Debt", "Total_EMI_per_month",
    "Amount_invested_monthly",
]
for col in NON_NEGATIVE:
    if col in df.columns:
        df.loc[df[col] < 0, col] = np.nan

In [ ]:
VALID_VALUES = {
    "Credit_Mix": {"Bad", "Standard", "Good"},

    "Occupation": {
        "Scientist", "Teacher", "Engineer", "Entrepreneur",
        "Developer", "Lawyer", "Media_Manager", "Doctor",
        "Journalist", "Manager", "Accountant", "Musician",
        "Mechanic", "Writer", "Architect",
    },

    "Payment_Behaviour" : {
        "High_spent_Small_value_payments",
        "Low_spent_Large_value_payments",
        "Low_spent_Medium_value_payments",
        "Low_spent_Small_value_payments",
        "High_spent_Large_value_payments",
        "High_spent_Medium_value_payments",
    },
}

print("Số NaN mới sinh ra sau khi lọc:")
for col, valid_set in VALID_VALUES.items():
    if col not in df.columns:
        continue

    before_nan   = df[col].isna().sum()
    before_dirty = (~df[col].isin(valid_set) & df[col].notna()).sum()

    # Thay giá trị không hợp lệ → NaN
    df[col] = df[col].where(df[col].isin(valid_set), other=np.nan)

    after_nan = df[col].isna().sum()
    print(f"  {col:<16} | giá trị lỗi: {before_dirty:>5} "
          f"| NaN trước: {before_nan:>5} → NaN sau: {after_nan:>5}")

Số NaN mới sinh ra sau khi lọc:
  Credit_Mix       | giá trị lỗi: 20195 | NaN trước:     0 → NaN sau: 20195
  Occupation       | giá trị lỗi:  7062 | NaN trước:     0 → NaN sau:  7062
  Payment_Behaviour | giá trị lỗi:  7600 | NaN trước:     0 → NaN sau:  7600


In [ ]:
missing_value = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({'Missing Values': missing_value, 'Percentage (%)': missing_percent})
print(missing_data[missing_data['Missing Values']> 0])

print(f"\n So luong dong trung lap: {df.duplicated().sum()}")

                         Missing Values  Percentage (%)
Age                                 886           0.886
Occupation                         7062           7.062
Monthly_Inhand_Salary             15002          15.002
Num_Bank_Accounts                    21           0.021
Num_of_Loan                        3876           3.876
Type_of_Loan                      11408          11.408
Num_of_Delayed_Payment             7646           7.646
Changed_Credit_Limit               2091           2.091
Num_Credit_Inquiries               1965           1.965
Credit_Mix                        20195          20.195
Credit_History_Age                 9030           9.030
Amount_invested_monthly            4479           4.479
Payment_Behaviour                  7600           7.600
Monthly_Balance                    1200           1.200

 So luong dong trung lap: 0


In [ ]:
def parse_credit_history_age(val):
    """'22 Years and 3 Months' → 267 (tháng)"""
    if pd.isna(val):
        return np.nan
    val = str(val)
    years  = re.search(r"(\d+)\s*Year",  val, re.IGNORECASE)
    months = re.search(r"(\d+)\s*Month", val, re.IGNORECASE)
    y = int(years.group(1))  if years  else 0
    m = int(months.group(1)) if months else 0
    return y * 12 + m

import re
df["Credit_History_Age"] = df["Credit_History_Age"].apply(parse_credit_history_age)

In [ ]:
def iqr_clip(series, factor=3.0):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return series.clip(lower=Q1 - factor * IQR, upper=Q3 + factor * IQR)

for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = iqr_clip(df[col])

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 0. Tiền xử lý Type_of_Loan trước (cột đặc biệt - multi-value)
# ============================================================
# Type_of_Loan chứa nhiều loại loan trong 1 cell, ví dụ:
# "Auto Loan, Credit-Builder Loan, Personal Loan"
# → cần xử lý riêng, không impute thô như categorical thông thường

def get_loan_mode_per_customer(df):
    """
    Với mỗi customer, tìm chuỗi Type_of_Loan xuất hiện nhiều nhất
    trong 8 tháng (trong số các tháng không bị NaN).
    """
    def mode_or_nan(x):
        valid = x.dropna()
        if len(valid) == 0:
            return np.nan
        return valid.mode()[0]

    return df.groupby("Customer_ID")["Type_of_Loan"].transform(mode_or_nan)


# ============================================================
# 1. Imputation cho các cột số
# ============================================================
NUMERIC_IMPUTE_COLS = [
    "Age",
    "Num_Bank_Accounts",
    "Num_of_Loan",
    "Changed_Credit_Limit",
    "Monthly_Inhand_Salary",
    "Num_of_Delayed_Payment",
    "Num_Credit_Inquiries",
    "Credit_History_Age",
    "Amount_invested_monthly",
    "Monthly_Balance",
]

def impute_numeric_by_customer(df, cols):
    """
    Điền NaN bằng median của chính Customer_ID đó.
    Không cần global fallback vì không có customer nào thiếu 8/8.
    """
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue

        before = df[col].isna().sum()

        df[col] = df.groupby("Customer_ID")[col].transform(
            lambda x: x.fillna(x.median())
        )

        after = df[col].isna().sum()
        print(f"  {col:<28} | filled: {before - after:>5} | còn lại: {after:>4}")

    return df


# ============================================================
# 2. Imputation cho Type_of_Loan (categorical + global fallback)
# ============================================================
def impute_type_of_loan(df):
    """
    Bước 1: Điền bằng mode của chính customer đó (các tháng có giá trị).
    Bước 2: 1426 customers thiếu 8/8 → điền bằng global mode.
    """
    df = df.copy()
    before = df["Type_of_Loan"].isna().sum()

    # Bước 1: customer-level mode
    df["Type_of_Loan"] = get_loan_mode_per_customer(df)

    # Bước 2: global fallback cho 1426 customers thiếu 8/8
    global_mode = df["Type_of_Loan"].mode()[0]
    remaining   = df["Type_of_Loan"].isna().sum()
    df["Type_of_Loan"] = df["Type_of_Loan"].fillna(global_mode)

    filled_customer = before - remaining
    filled_global   = remaining

    print(f"  {'Type_of_Loan':<28} | "
          f"customer-fill: {filled_customer:>5} | "
          f"global-fill: {filled_global:>5} (global_mode='{global_mode}')")

    return df


# ============================================================
# 3. Chạy toàn bộ pipeline imputation
# ============================================================
def run_imputation(df):
    print(">>> BẮT ĐẦU IMPUTATION\n")

    print("[1] Numeric — customer median:")
    df = impute_numeric_by_customer(df, NUMERIC_IMPUTE_COLS)

    print("\n[2] Categorical — customer mode + global fallback:")
    df = impute_type_of_loan(df)

    # Kiểm tra tổng kết
    remaining_missing = df[NUMERIC_IMPUTE_COLS + ["Type_of_Loan"]].isna().sum()
    remaining_missing = remaining_missing[remaining_missing > 0]

    print("\n>>> KẾT QUẢ SAU IMPUTATION:")
    if len(remaining_missing) == 0:
        print("  ✅ Không còn missing value trong các cột đã xử lý.")
    else:
        print(remaining_missing)

    return df

df = run_imputation(df)

>>> BẮT ĐẦU IMPUTATION

[1] Numeric — customer median:
  Age                          | filled:   886 | còn lại:    0
  Num_Bank_Accounts            | filled:    21 | còn lại:    0
  Num_of_Loan                  | filled:  3876 | còn lại:    0
  Changed_Credit_Limit         | filled:  2091 | còn lại:    0
  Monthly_Inhand_Salary        | filled: 15002 | còn lại:    0
  Num_of_Delayed_Payment       | filled:  7646 | còn lại:    0
  Num_Credit_Inquiries         | filled:  1965 | còn lại:    0
  Credit_History_Age           | filled:  9030 | còn lại:    0
  Amount_invested_monthly      | filled:  4479 | còn lại:    0
  Monthly_Balance              | filled:  1200 | còn lại:    0

[2] Categorical — customer mode + global fallback:
  Type_of_Loan                 | customer-fill:     0 | global-fill: 11408 (global_mode='Not Specified')

>>> KẾT QUẢ SAU IMPUTATION:
  ✅ Không còn missing value trong các cột đã xử lý.


In [ ]:
def impute_categorical_by_customer(df, cols):
    """
    Bước 1: mode của chính Customer_ID đó.
    Bước 2: global mode fallback nếu customer thiếu 8/8.
    """
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue

        before = df[col].isna().sum()

        # Bước 1: customer-level mode
        def fill_mode(x):
            valid = x.dropna()
            if len(valid) == 0:
                return x           # toàn NaN → để fallback xử lý
            return x.fillna(valid.mode()[0])

        df[col] = df.groupby("Customer_ID")[col].transform(fill_mode)

        # Bước 2: global mode fallback
        after_step1   = df[col].isna().sum()
        global_mode   = df[col].mode()[0]
        df[col]       = df[col].fillna(global_mode)
        after_step2   = df[col].isna().sum()

        cust_filled   = before - after_step1
        global_filled = after_step1 - after_step2

        note = f" | ⚠ global fallback: {global_filled} (mode='{global_mode}')" \
               if global_filled > 0 else ""
        print(f"  {col:<16} filled: {cust_filled:>5} (customer){note} "
              f"| còn lại: {after_step2}")

    return df


print("Imputing Occupation & Credit_Mix:")
df = impute_categorical_by_customer(df, list(VALID_VALUES.keys()))

Imputing Occupation & Credit_Mix:
  Credit_Mix       filled: 20195 (customer) | còn lại: 0
  Occupation       filled:  7062 (customer) | còn lại: 0
  Payment_Behaviour filled:  7600 (customer) | còn lại: 0


In [ ]:
missing_value = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({'Missing Values': missing_value, 'Percentage (%)': missing_percent})
print(missing_data[missing_data['Missing Values']> 0])

print(f"\n So luong dong trung lap: {df.duplicated().sum()}")

Empty DataFrame
Columns: [Missing Values, Percentage (%)]
Index: []

 So luong dong trung lap: 0


split and Enconde